# Lab

Build and maintain the facilities team's six-room route graph. Each task extends the same `RouteGraph` design: create vertices, store routes, close routes or rooms, and verify the result against a matrix.

Every cell includes its own setup and class. Carry your completed methods forward explicitly. Use Edit to change a draft and Run to check it. Sign in before submitting; multiple submissions are allowed and the highest score is kept.

## 1. Preserve Every Room

```{index} Preserve Every Room
```

Complete `AddVertex` so a new label installs an empty neighbor map and a repeated label returns false. Do not wait for an edge to create a room. Use ordinal, case-sensitive equality.

Expected results: duplicate false, six vertices, zero edges, and Remote degree zero. An isolate is an existing room, not an unknown label.

In [ ]:
using System;
using System.Collections.Generic;
using System.Linq;

var graph = new RouteGraph();
foreach (string room in new[] { "Lobby", "Hall", "Lab", "Office", "Storage", "Remote" }) graph.AddVertex(room);
Console.WriteLine($"duplicate added={graph.AddVertex("Remote")}");
Console.WriteLine($"vertices={graph.VertexCount}, edges={graph.EdgeCount}, Remote degree={graph.Degree("Remote")}");

public class RouteGraph
{
    private readonly Dictionary<string, Dictionary<string, int>> adjacency
        = new(StringComparer.Ordinal);
    public int VertexCount => adjacency.Count;
    public int EdgeCount { get; private set; }

    public bool AddVertex(string vertex)
    {
        // Your code starts here.
        /* TODO */
        // Your code ends here.
    }

    public bool SetEdge(string a, string b, int minutes)
    {
        if (a == b) throw new ArgumentException("Self-loops are not allowed.");
        if (minutes < 0) throw new ArgumentOutOfRangeException(nameof(minutes));
        RequireVertex(a);
        RequireVertex(b);
        bool added = !adjacency[a].ContainsKey(b);
        adjacency[a][b] = minutes;
        adjacency[b][a] = minutes;
        if (added) EdgeCount++;
        return added;
    }

    public bool TryGetWeight(string a, string b, out int minutes)
    {
        minutes = default;
        return adjacency.TryGetValue(a, out var neighbors)
            && neighbors.TryGetValue(b, out minutes);
    }

    public int Degree(string vertex)
    {
        RequireVertex(vertex);
        return adjacency[vertex].Count;
    }

    public KeyValuePair<string, int>[] Neighbors(string vertex)
    {
        RequireVertex(vertex);
        return adjacency[vertex].ToArray();
    }

    public string[] Vertices() => adjacency.Keys.ToArray();

    public bool RemoveEdge(string a, string b)
    {
        if (!adjacency.TryGetValue(a, out var neighbors) || !neighbors.Remove(b)) return false;
        adjacency[b].Remove(a);
        EdgeCount--;
        return true;
    }

    public bool RemoveVertex(string vertex)
    {
        if (!adjacency.TryGetValue(vertex, out var neighbors)) return false;
        foreach (string neighbor in neighbors.Keys) adjacency[neighbor].Remove(vertex);
        EdgeCount -= neighbors.Count;
        adjacency.Remove(vertex);
        return true;
    }

    public bool IsValid()
    {
        long degreeSum = 0;
        foreach (var (vertex, neighbors) in adjacency)
        {
            degreeSum += neighbors.Count;
            foreach (var (neighbor, minutes) in neighbors)
            {
                if (vertex == neighbor || minutes < 0
                    || !adjacency.TryGetValue(neighbor, out var reverse)
                    || !reverse.TryGetValue(vertex, out int reverseMinutes)
                    || reverseMinutes != minutes) return false;
            }
        }
        return degreeSum == 2L * EdgeCount;
    }

    private void RequireVertex(string vertex)
    {
        if (!adjacency.ContainsKey(vertex)) throw new KeyNotFoundException($"Unknown vertex: {vertex}");
    }
}

In [1]:
// Solution
using System;
using System.Collections.Generic;
using System.Linq;

var graph = new RouteGraph();
foreach (string room in new[] { "Lobby", "Hall", "Lab", "Office", "Storage", "Remote" }) graph.AddVertex(room);
Console.WriteLine($"duplicate added={graph.AddVertex("Remote")}");
Console.WriteLine($"vertices={graph.VertexCount}, edges={graph.EdgeCount}, Remote degree={graph.Degree("Remote")}");

public class RouteGraph
{
    private readonly Dictionary<string, Dictionary<string, int>> adjacency
        = new(StringComparer.Ordinal);
    public int VertexCount => adjacency.Count;
    public int EdgeCount { get; private set; }

    public bool AddVertex(string vertex)
        => adjacency.TryAdd(vertex, new Dictionary<string, int>(StringComparer.Ordinal));

    public bool SetEdge(string a, string b, int minutes)
    {
        if (a == b) throw new ArgumentException("Self-loops are not allowed.");
        if (minutes < 0) throw new ArgumentOutOfRangeException(nameof(minutes));
        RequireVertex(a);
        RequireVertex(b);
        bool added = !adjacency[a].ContainsKey(b);
        adjacency[a][b] = minutes;
        adjacency[b][a] = minutes;
        if (added) EdgeCount++;
        return added;
    }

    public bool TryGetWeight(string a, string b, out int minutes)
    {
        minutes = default;
        return adjacency.TryGetValue(a, out var neighbors)
            && neighbors.TryGetValue(b, out minutes);
    }

    public int Degree(string vertex)
    {
        RequireVertex(vertex);
        return adjacency[vertex].Count;
    }

    public KeyValuePair<string, int>[] Neighbors(string vertex)
    {
        RequireVertex(vertex);
        return adjacency[vertex].ToArray();
    }

    public string[] Vertices() => adjacency.Keys.ToArray();

    public bool RemoveEdge(string a, string b)
    {
        if (!adjacency.TryGetValue(a, out var neighbors) || !neighbors.Remove(b)) return false;
        adjacency[b].Remove(a);
        EdgeCount--;
        return true;
    }

    public bool RemoveVertex(string vertex)
    {
        if (!adjacency.TryGetValue(vertex, out var neighbors)) return false;
        foreach (string neighbor in neighbors.Keys) adjacency[neighbor].Remove(vertex);
        EdgeCount -= neighbors.Count;
        adjacency.Remove(vertex);
        return true;
    }

    public bool IsValid()
    {
        long degreeSum = 0;
        foreach (var (vertex, neighbors) in adjacency)
        {
            degreeSum += neighbors.Count;
            foreach (var (neighbor, minutes) in neighbors)
            {
                if (vertex == neighbor || minutes < 0
                    || !adjacency.TryGetValue(neighbor, out var reverse)
                    || !reverse.TryGetValue(vertex, out int reverseMinutes)
                    || reverseMinutes != minutes) return false;
            }
        }
        return degreeSum == 2L * EdgeCount;
    }

    private void RequireVertex(string vertex)
    {
        if (!adjacency.ContainsKey(vertex)) throw new KeyNotFoundException($"Unknown vertex: {vertex}");
    }
}

duplicate added=False
vertices=6, edges=0, Remote degree=0


## 2. Store and Replace Two-Way Routes

```{index} Store and Replace Two-Way Routes
```

Complete `SetEdge`. Reject self-loops and negative minutes, require both endpoints, and update both reciprocal weights. Increment EdgeCount only when the logical edge is new. Carry your vertex method forward.

Expected replacement false, four edges, reverse Lab–Hall weight five, and Office–Storage present with weight zero. Explain why testing `weight != 0` would lose a real edge.

In [ ]:
using System;
using System.Collections.Generic;
using System.Linq;

var graph = new RouteGraph();
foreach (string room in new[] { "Lobby", "Hall", "Lab", "Office", "Storage", "Remote" }) graph.AddVertex(room);
graph.SetEdge("Lobby", "Hall", 4);
graph.SetEdge("Hall", "Lab", 2);
graph.SetEdge("Hall", "Office", 3);
graph.SetEdge("Office", "Storage", 0);
Console.WriteLine($"replacement inserted={graph.SetEdge("Hall", "Lab", 5)}, edges={graph.EdgeCount}");
Console.WriteLine($"reverse found={graph.TryGetWeight("Lab", "Hall", out int minutes)}, minutes={minutes}");
Console.WriteLine($"zero edge found={graph.TryGetWeight("Office", "Storage", out int zero)}, minutes={zero}");
Console.WriteLine($"valid={graph.IsValid()}");

public class RouteGraph
{
    private readonly Dictionary<string, Dictionary<string, int>> adjacency
        = new(StringComparer.Ordinal);
    public int VertexCount => adjacency.Count;
    public int EdgeCount { get; private set; }

    public bool AddVertex(string vertex)
        => adjacency.TryAdd(vertex, new Dictionary<string, int>(StringComparer.Ordinal));

    public bool SetEdge(string a, string b, int minutes)
    {
        // Your code starts here.
        /* TODO */
        // Your code ends here.
    }

    public bool TryGetWeight(string a, string b, out int minutes)
    {
        minutes = default;
        return adjacency.TryGetValue(a, out var neighbors)
            && neighbors.TryGetValue(b, out minutes);
    }

    public int Degree(string vertex)
    {
        RequireVertex(vertex);
        return adjacency[vertex].Count;
    }

    public KeyValuePair<string, int>[] Neighbors(string vertex)
    {
        RequireVertex(vertex);
        return adjacency[vertex].ToArray();
    }

    public string[] Vertices() => adjacency.Keys.ToArray();

    public bool RemoveEdge(string a, string b)
    {
        if (!adjacency.TryGetValue(a, out var neighbors) || !neighbors.Remove(b)) return false;
        adjacency[b].Remove(a);
        EdgeCount--;
        return true;
    }

    public bool RemoveVertex(string vertex)
    {
        if (!adjacency.TryGetValue(vertex, out var neighbors)) return false;
        foreach (string neighbor in neighbors.Keys) adjacency[neighbor].Remove(vertex);
        EdgeCount -= neighbors.Count;
        adjacency.Remove(vertex);
        return true;
    }

    public bool IsValid()
    {
        long degreeSum = 0;
        foreach (var (vertex, neighbors) in adjacency)
        {
            degreeSum += neighbors.Count;
            foreach (var (neighbor, minutes) in neighbors)
            {
                if (vertex == neighbor || minutes < 0
                    || !adjacency.TryGetValue(neighbor, out var reverse)
                    || !reverse.TryGetValue(vertex, out int reverseMinutes)
                    || reverseMinutes != minutes) return false;
            }
        }
        return degreeSum == 2L * EdgeCount;
    }

    private void RequireVertex(string vertex)
    {
        if (!adjacency.ContainsKey(vertex)) throw new KeyNotFoundException($"Unknown vertex: {vertex}");
    }
}

In [1]:
// Solution
using System;
using System.Collections.Generic;
using System.Linq;

var graph = new RouteGraph();
foreach (string room in new[] { "Lobby", "Hall", "Lab", "Office", "Storage", "Remote" }) graph.AddVertex(room);
graph.SetEdge("Lobby", "Hall", 4);
graph.SetEdge("Hall", "Lab", 2);
graph.SetEdge("Hall", "Office", 3);
graph.SetEdge("Office", "Storage", 0);
Console.WriteLine($"replacement inserted={graph.SetEdge("Hall", "Lab", 5)}, edges={graph.EdgeCount}");
Console.WriteLine($"reverse found={graph.TryGetWeight("Lab", "Hall", out int minutes)}, minutes={minutes}");
Console.WriteLine($"zero edge found={graph.TryGetWeight("Office", "Storage", out int zero)}, minutes={zero}");
Console.WriteLine($"valid={graph.IsValid()}");

public class RouteGraph
{
    private readonly Dictionary<string, Dictionary<string, int>> adjacency
        = new(StringComparer.Ordinal);
    public int VertexCount => adjacency.Count;
    public int EdgeCount { get; private set; }

    public bool AddVertex(string vertex)
        => adjacency.TryAdd(vertex, new Dictionary<string, int>(StringComparer.Ordinal));

    public bool SetEdge(string a, string b, int minutes)
    {
        if (a == b) throw new ArgumentException("Self-loops are not allowed.");
        if (minutes < 0) throw new ArgumentOutOfRangeException(nameof(minutes));
        RequireVertex(a);
        RequireVertex(b);
        bool added = !adjacency[a].ContainsKey(b);
        adjacency[a][b] = minutes;
        adjacency[b][a] = minutes;
        if (added) EdgeCount++;
        return added;
    }

    public bool TryGetWeight(string a, string b, out int minutes)
    {
        minutes = default;
        return adjacency.TryGetValue(a, out var neighbors)
            && neighbors.TryGetValue(b, out minutes);
    }

    public int Degree(string vertex)
    {
        RequireVertex(vertex);
        return adjacency[vertex].Count;
    }

    public KeyValuePair<string, int>[] Neighbors(string vertex)
    {
        RequireVertex(vertex);
        return adjacency[vertex].ToArray();
    }

    public string[] Vertices() => adjacency.Keys.ToArray();

    public bool RemoveEdge(string a, string b)
    {
        if (!adjacency.TryGetValue(a, out var neighbors) || !neighbors.Remove(b)) return false;
        adjacency[b].Remove(a);
        EdgeCount--;
        return true;
    }

    public bool RemoveVertex(string vertex)
    {
        if (!adjacency.TryGetValue(vertex, out var neighbors)) return false;
        foreach (string neighbor in neighbors.Keys) adjacency[neighbor].Remove(vertex);
        EdgeCount -= neighbors.Count;
        adjacency.Remove(vertex);
        return true;
    }

    public bool IsValid()
    {
        long degreeSum = 0;
        foreach (var (vertex, neighbors) in adjacency)
        {
            degreeSum += neighbors.Count;
            foreach (var (neighbor, minutes) in neighbors)
            {
                if (vertex == neighbor || minutes < 0
                    || !adjacency.TryGetValue(neighbor, out var reverse)
                    || !reverse.TryGetValue(vertex, out int reverseMinutes)
                    || reverseMinutes != minutes) return false;
            }
        }
        return degreeSum == 2L * EdgeCount;
    }

    private void RequireVertex(string vertex)
    {
        if (!adjacency.ContainsKey(vertex)) throw new KeyNotFoundException($"Unknown vertex: {vertex}");
    }
}

replacement inserted=False, edges=4
reverse found=True, minutes=5
zero edge found=True, minutes=0
valid=True


## 3. Close One Passageway

```{index} Close One Passageway
```

Complete `RemoveEdge` so one successful removal deletes both directions and reduces EdgeCount once. A reversed repeat removal returns false. Keep all rooms present.

Expected six vertices, three edges, Hall degree two, and both direct lookups false. Office–Storage must remain. Why does leaving the reverse entry violate the graph's contract?

In [ ]:
using System;
using System.Collections.Generic;
using System.Linq;

var graph = new RouteGraph();
foreach (string room in new[] { "Lobby", "Hall", "Lab", "Office", "Storage", "Remote" }) graph.AddVertex(room);
graph.SetEdge("Lobby", "Hall", 4);
graph.SetEdge("Hall", "Lab", 2);
graph.SetEdge("Hall", "Office", 3);
graph.SetEdge("Office", "Storage", 0);
Console.WriteLine($"removed={graph.RemoveEdge("Hall", "Office")}");
Console.WriteLine($"removed again={graph.RemoveEdge("Office", "Hall")}");
Console.WriteLine($"forward exists={graph.TryGetWeight("Hall", "Office", out _)}, reverse exists={graph.TryGetWeight("Office", "Hall", out _)}");
Console.WriteLine($"vertices={graph.VertexCount}, edges={graph.EdgeCount}, Hall degree={graph.Degree("Hall")}, valid={graph.IsValid()}");

public class RouteGraph
{
    private readonly Dictionary<string, Dictionary<string, int>> adjacency
        = new(StringComparer.Ordinal);
    public int VertexCount => adjacency.Count;
    public int EdgeCount { get; private set; }

    public bool AddVertex(string vertex)
        => adjacency.TryAdd(vertex, new Dictionary<string, int>(StringComparer.Ordinal));

    public bool SetEdge(string a, string b, int minutes)
    {
        if (a == b) throw new ArgumentException("Self-loops are not allowed.");
        if (minutes < 0) throw new ArgumentOutOfRangeException(nameof(minutes));
        RequireVertex(a);
        RequireVertex(b);
        bool added = !adjacency[a].ContainsKey(b);
        adjacency[a][b] = minutes;
        adjacency[b][a] = minutes;
        if (added) EdgeCount++;
        return added;
    }

    public bool TryGetWeight(string a, string b, out int minutes)
    {
        minutes = default;
        return adjacency.TryGetValue(a, out var neighbors)
            && neighbors.TryGetValue(b, out minutes);
    }

    public int Degree(string vertex)
    {
        RequireVertex(vertex);
        return adjacency[vertex].Count;
    }

    public KeyValuePair<string, int>[] Neighbors(string vertex)
    {
        RequireVertex(vertex);
        return adjacency[vertex].ToArray();
    }

    public string[] Vertices() => adjacency.Keys.ToArray();

    public bool RemoveEdge(string a, string b)
    {
        // Your code starts here.
        /* TODO */
        // Your code ends here.
    }

    public bool RemoveVertex(string vertex)
    {
        if (!adjacency.TryGetValue(vertex, out var neighbors)) return false;
        foreach (string neighbor in neighbors.Keys) adjacency[neighbor].Remove(vertex);
        EdgeCount -= neighbors.Count;
        adjacency.Remove(vertex);
        return true;
    }

    public bool IsValid()
    {
        long degreeSum = 0;
        foreach (var (vertex, neighbors) in adjacency)
        {
            degreeSum += neighbors.Count;
            foreach (var (neighbor, minutes) in neighbors)
            {
                if (vertex == neighbor || minutes < 0
                    || !adjacency.TryGetValue(neighbor, out var reverse)
                    || !reverse.TryGetValue(vertex, out int reverseMinutes)
                    || reverseMinutes != minutes) return false;
            }
        }
        return degreeSum == 2L * EdgeCount;
    }

    private void RequireVertex(string vertex)
    {
        if (!adjacency.ContainsKey(vertex)) throw new KeyNotFoundException($"Unknown vertex: {vertex}");
    }
}

In [1]:
// Solution
using System;
using System.Collections.Generic;
using System.Linq;

var graph = new RouteGraph();
foreach (string room in new[] { "Lobby", "Hall", "Lab", "Office", "Storage", "Remote" }) graph.AddVertex(room);
graph.SetEdge("Lobby", "Hall", 4);
graph.SetEdge("Hall", "Lab", 2);
graph.SetEdge("Hall", "Office", 3);
graph.SetEdge("Office", "Storage", 0);
Console.WriteLine($"removed={graph.RemoveEdge("Hall", "Office")}");
Console.WriteLine($"removed again={graph.RemoveEdge("Office", "Hall")}");
Console.WriteLine($"forward exists={graph.TryGetWeight("Hall", "Office", out _)}, reverse exists={graph.TryGetWeight("Office", "Hall", out _)}");
Console.WriteLine($"vertices={graph.VertexCount}, edges={graph.EdgeCount}, Hall degree={graph.Degree("Hall")}, valid={graph.IsValid()}");

public class RouteGraph
{
    private readonly Dictionary<string, Dictionary<string, int>> adjacency
        = new(StringComparer.Ordinal);
    public int VertexCount => adjacency.Count;
    public int EdgeCount { get; private set; }

    public bool AddVertex(string vertex)
        => adjacency.TryAdd(vertex, new Dictionary<string, int>(StringComparer.Ordinal));

    public bool SetEdge(string a, string b, int minutes)
    {
        if (a == b) throw new ArgumentException("Self-loops are not allowed.");
        if (minutes < 0) throw new ArgumentOutOfRangeException(nameof(minutes));
        RequireVertex(a);
        RequireVertex(b);
        bool added = !adjacency[a].ContainsKey(b);
        adjacency[a][b] = minutes;
        adjacency[b][a] = minutes;
        if (added) EdgeCount++;
        return added;
    }

    public bool TryGetWeight(string a, string b, out int minutes)
    {
        minutes = default;
        return adjacency.TryGetValue(a, out var neighbors)
            && neighbors.TryGetValue(b, out minutes);
    }

    public int Degree(string vertex)
    {
        RequireVertex(vertex);
        return adjacency[vertex].Count;
    }

    public KeyValuePair<string, int>[] Neighbors(string vertex)
    {
        RequireVertex(vertex);
        return adjacency[vertex].ToArray();
    }

    public string[] Vertices() => adjacency.Keys.ToArray();

    public bool RemoveEdge(string a, string b)
    {
        if (!adjacency.TryGetValue(a, out var neighbors) || !neighbors.Remove(b)) return false;
        adjacency[b].Remove(a);
        EdgeCount--;
        return true;
    }

    public bool RemoveVertex(string vertex)
    {
        if (!adjacency.TryGetValue(vertex, out var neighbors)) return false;
        foreach (string neighbor in neighbors.Keys) adjacency[neighbor].Remove(vertex);
        EdgeCount -= neighbors.Count;
        adjacency.Remove(vertex);
        return true;
    }

    public bool IsValid()
    {
        long degreeSum = 0;
        foreach (var (vertex, neighbors) in adjacency)
        {
            degreeSum += neighbors.Count;
            foreach (var (neighbor, minutes) in neighbors)
            {
                if (vertex == neighbor || minutes < 0
                    || !adjacency.TryGetValue(neighbor, out var reverse)
                    || !reverse.TryGetValue(vertex, out int reverseMinutes)
                    || reverseMinutes != minutes) return false;
            }
        }
        return degreeSum == 2L * EdgeCount;
    }

    private void RequireVertex(string vertex)
    {
        if (!adjacency.ContainsKey(vertex)) throw new KeyNotFoundException($"Unknown vertex: {vertex}");
    }
}

removed=True
removed again=False
forward exists=False, reverse exists=False
vertices=6, edges=3, Hall degree=2, valid=True


## 4. Close and Reopen a Room

```{index} Close and Reopen a Room
```

Complete `RemoveVertex`. Remove every incoming reciprocal reference, subtract the removed room's degree from EdgeCount, and delete its vertex entry. Preserve rooms that become isolates. Reuse the completed operations from Tasks 1–3.

Expected five vertices and one edge after removing Hall; Lobby and Lab have degree zero. Remote stays present. Readding Hall creates an isolate and does not restore its old routes. Explain why the code can enumerate Hall's map while changing its neighbors' maps.

In [ ]:
using System;
using System.Collections.Generic;
using System.Linq;

var graph = new RouteGraph();
foreach (string room in new[] { "Lobby", "Hall", "Lab", "Office", "Storage", "Remote" }) graph.AddVertex(room);
graph.SetEdge("Lobby", "Hall", 4);
graph.SetEdge("Hall", "Lab", 2);
graph.SetEdge("Hall", "Office", 3);
graph.SetEdge("Office", "Storage", 0);
Console.WriteLine($"removed Hall={graph.RemoveVertex("Hall")}");
Console.WriteLine($"vertices={graph.VertexCount}, edges={graph.EdgeCount}, Lobby degree={graph.Degree("Lobby")}, Lab degree={graph.Degree("Lab")}");
Console.WriteLine($"removed again={graph.RemoveVertex("Hall")}, Remote degree={graph.Degree("Remote")}, valid={graph.IsValid()}");
graph.AddVertex("Hall");
Console.WriteLine($"readded Hall degree={graph.Degree("Hall")}, edges={graph.EdgeCount}");

public class RouteGraph
{
    private readonly Dictionary<string, Dictionary<string, int>> adjacency
        = new(StringComparer.Ordinal);
    public int VertexCount => adjacency.Count;
    public int EdgeCount { get; private set; }

    public bool AddVertex(string vertex)
        => adjacency.TryAdd(vertex, new Dictionary<string, int>(StringComparer.Ordinal));

    public bool SetEdge(string a, string b, int minutes)
    {
        if (a == b) throw new ArgumentException("Self-loops are not allowed.");
        if (minutes < 0) throw new ArgumentOutOfRangeException(nameof(minutes));
        RequireVertex(a);
        RequireVertex(b);
        bool added = !adjacency[a].ContainsKey(b);
        adjacency[a][b] = minutes;
        adjacency[b][a] = minutes;
        if (added) EdgeCount++;
        return added;
    }

    public bool TryGetWeight(string a, string b, out int minutes)
    {
        minutes = default;
        return adjacency.TryGetValue(a, out var neighbors)
            && neighbors.TryGetValue(b, out minutes);
    }

    public int Degree(string vertex)
    {
        RequireVertex(vertex);
        return adjacency[vertex].Count;
    }

    public KeyValuePair<string, int>[] Neighbors(string vertex)
    {
        RequireVertex(vertex);
        return adjacency[vertex].ToArray();
    }

    public string[] Vertices() => adjacency.Keys.ToArray();

    public bool RemoveEdge(string a, string b)
    {
        if (!adjacency.TryGetValue(a, out var neighbors) || !neighbors.Remove(b)) return false;
        adjacency[b].Remove(a);
        EdgeCount--;
        return true;
    }

    public bool RemoveVertex(string vertex)
    {
        // Your code starts here.
        /* TODO */
        // Your code ends here.
    }

    public bool IsValid()
    {
        long degreeSum = 0;
        foreach (var (vertex, neighbors) in adjacency)
        {
            degreeSum += neighbors.Count;
            foreach (var (neighbor, minutes) in neighbors)
            {
                if (vertex == neighbor || minutes < 0
                    || !adjacency.TryGetValue(neighbor, out var reverse)
                    || !reverse.TryGetValue(vertex, out int reverseMinutes)
                    || reverseMinutes != minutes) return false;
            }
        }
        return degreeSum == 2L * EdgeCount;
    }

    private void RequireVertex(string vertex)
    {
        if (!adjacency.ContainsKey(vertex)) throw new KeyNotFoundException($"Unknown vertex: {vertex}");
    }
}

In [1]:
// Solution
using System;
using System.Collections.Generic;
using System.Linq;

var graph = new RouteGraph();
foreach (string room in new[] { "Lobby", "Hall", "Lab", "Office", "Storage", "Remote" }) graph.AddVertex(room);
graph.SetEdge("Lobby", "Hall", 4);
graph.SetEdge("Hall", "Lab", 2);
graph.SetEdge("Hall", "Office", 3);
graph.SetEdge("Office", "Storage", 0);
Console.WriteLine($"removed Hall={graph.RemoveVertex("Hall")}");
Console.WriteLine($"vertices={graph.VertexCount}, edges={graph.EdgeCount}, Lobby degree={graph.Degree("Lobby")}, Lab degree={graph.Degree("Lab")}");
Console.WriteLine($"removed again={graph.RemoveVertex("Hall")}, Remote degree={graph.Degree("Remote")}, valid={graph.IsValid()}");
graph.AddVertex("Hall");
Console.WriteLine($"readded Hall degree={graph.Degree("Hall")}, edges={graph.EdgeCount}");

public class RouteGraph
{
    private readonly Dictionary<string, Dictionary<string, int>> adjacency
        = new(StringComparer.Ordinal);
    public int VertexCount => adjacency.Count;
    public int EdgeCount { get; private set; }

    public bool AddVertex(string vertex)
        => adjacency.TryAdd(vertex, new Dictionary<string, int>(StringComparer.Ordinal));

    public bool SetEdge(string a, string b, int minutes)
    {
        if (a == b) throw new ArgumentException("Self-loops are not allowed.");
        if (minutes < 0) throw new ArgumentOutOfRangeException(nameof(minutes));
        RequireVertex(a);
        RequireVertex(b);
        bool added = !adjacency[a].ContainsKey(b);
        adjacency[a][b] = minutes;
        adjacency[b][a] = minutes;
        if (added) EdgeCount++;
        return added;
    }

    public bool TryGetWeight(string a, string b, out int minutes)
    {
        minutes = default;
        return adjacency.TryGetValue(a, out var neighbors)
            && neighbors.TryGetValue(b, out minutes);
    }

    public int Degree(string vertex)
    {
        RequireVertex(vertex);
        return adjacency[vertex].Count;
    }

    public KeyValuePair<string, int>[] Neighbors(string vertex)
    {
        RequireVertex(vertex);
        return adjacency[vertex].ToArray();
    }

    public string[] Vertices() => adjacency.Keys.ToArray();

    public bool RemoveEdge(string a, string b)
    {
        if (!adjacency.TryGetValue(a, out var neighbors) || !neighbors.Remove(b)) return false;
        adjacency[b].Remove(a);
        EdgeCount--;
        return true;
    }

    public bool RemoveVertex(string vertex)
    {
        if (!adjacency.TryGetValue(vertex, out var neighbors)) return false;
        foreach (string neighbor in neighbors.Keys) adjacency[neighbor].Remove(vertex);
        EdgeCount -= neighbors.Count;
        adjacency.Remove(vertex);
        return true;
    }

    public bool IsValid()
    {
        long degreeSum = 0;
        foreach (var (vertex, neighbors) in adjacency)
        {
            degreeSum += neighbors.Count;
            foreach (var (neighbor, minutes) in neighbors)
            {
                if (vertex == neighbor || minutes < 0
                    || !adjacency.TryGetValue(neighbor, out var reverse)
                    || !reverse.TryGetValue(vertex, out int reverseMinutes)
                    || reverseMinutes != minutes) return false;
            }
        }
        return degreeSum == 2L * EdgeCount;
    }

    private void RequireVertex(string vertex)
    {
        if (!adjacency.ContainsKey(vertex)) throw new KeyNotFoundException($"Unknown vertex: {vertex}");
    }
}

removed Hall=True
vertices=5, edges=1, Lobby degree=0, Lab degree=0
removed again=False, Remote degree=0, valid=True
readded Hall degree=0, edges=1


## 5. Compare Two Representations

```{index} Compare Two Representations
```

Complete `ToMatrix` using the supplied label snapshot, which contains every distinct graph vertex. Initialize a nullable matrix, map labels to indexes, and copy each stored adjacency entry, including zero values. Use your graph methods from Tasks 1–4.

The supplied checks compare every pair for presence and weight, verify symmetry, and check degree sum. Remote must retain a row/column even though it has no edges. Describe a defect that would pass symmetry but fail the full membership check, such as forgetting an entire logical edge.

In [ ]:
using System;
using System.Collections.Generic;
using System.Linq;

var graph = new RouteGraph();
foreach (string room in new[] { "Lobby", "Hall", "Lab", "Office", "Storage", "Remote" }) graph.AddVertex(room);
graph.SetEdge("Lobby", "Hall", 4);
graph.SetEdge("Hall", "Lab", 2);
graph.SetEdge("Hall", "Office", 3);
graph.SetEdge("Office", "Storage", 0);
string[] labels = graph.Vertices().OrderBy(r => r, StringComparer.Ordinal).ToArray();
int?[,] matrix = ToMatrix(graph, labels);
for (int i = 0; i < labels.Length; i++)
    for (int j = 0; j < labels.Length; j++)
    {
        bool exists = graph.TryGetWeight(labels[i], labels[j], out int weight);
        if (matrix[i, j].HasValue != exists || (exists && matrix[i, j] != weight))
            throw new Exception("Matrix mismatch.");
        if (matrix[i, j] != matrix[j, i]) throw new Exception("Symmetry mismatch.");
    }
int degreeSum = graph.Vertices().Sum(v => graph.Degree(v));
if (degreeSum != 2 * graph.EdgeCount || !graph.IsValid()) throw new Exception("Invariant mismatch.");
Console.WriteLine("All-pair presence, weights, symmetry, isolates, and degree sum passed.");

int?[,] ToMatrix(RouteGraph graph, string[] labels)
{
        // Your code starts here.
        /* TODO */
        // Your code ends here.
    }

public class RouteGraph
{
    private readonly Dictionary<string, Dictionary<string, int>> adjacency
        = new(StringComparer.Ordinal);
    public int VertexCount => adjacency.Count;
    public int EdgeCount { get; private set; }

    public bool AddVertex(string vertex)
        => adjacency.TryAdd(vertex, new Dictionary<string, int>(StringComparer.Ordinal));

    public bool SetEdge(string a, string b, int minutes)
    {
        if (a == b) throw new ArgumentException("Self-loops are not allowed.");
        if (minutes < 0) throw new ArgumentOutOfRangeException(nameof(minutes));
        RequireVertex(a);
        RequireVertex(b);
        bool added = !adjacency[a].ContainsKey(b);
        adjacency[a][b] = minutes;
        adjacency[b][a] = minutes;
        if (added) EdgeCount++;
        return added;
    }

    public bool TryGetWeight(string a, string b, out int minutes)
    {
        minutes = default;
        return adjacency.TryGetValue(a, out var neighbors)
            && neighbors.TryGetValue(b, out minutes);
    }

    public int Degree(string vertex)
    {
        RequireVertex(vertex);
        return adjacency[vertex].Count;
    }

    public KeyValuePair<string, int>[] Neighbors(string vertex)
    {
        RequireVertex(vertex);
        return adjacency[vertex].ToArray();
    }

    public string[] Vertices() => adjacency.Keys.ToArray();

    public bool RemoveEdge(string a, string b)
    {
        if (!adjacency.TryGetValue(a, out var neighbors) || !neighbors.Remove(b)) return false;
        adjacency[b].Remove(a);
        EdgeCount--;
        return true;
    }

    public bool RemoveVertex(string vertex)
    {
        if (!adjacency.TryGetValue(vertex, out var neighbors)) return false;
        foreach (string neighbor in neighbors.Keys) adjacency[neighbor].Remove(vertex);
        EdgeCount -= neighbors.Count;
        adjacency.Remove(vertex);
        return true;
    }

    public bool IsValid()
    {
        long degreeSum = 0;
        foreach (var (vertex, neighbors) in adjacency)
        {
            degreeSum += neighbors.Count;
            foreach (var (neighbor, minutes) in neighbors)
            {
                if (vertex == neighbor || minutes < 0
                    || !adjacency.TryGetValue(neighbor, out var reverse)
                    || !reverse.TryGetValue(vertex, out int reverseMinutes)
                    || reverseMinutes != minutes) return false;
            }
        }
        return degreeSum == 2L * EdgeCount;
    }

    private void RequireVertex(string vertex)
    {
        if (!adjacency.ContainsKey(vertex)) throw new KeyNotFoundException($"Unknown vertex: {vertex}");
    }
}

In [1]:
// Solution
using System;
using System.Collections.Generic;
using System.Linq;

var graph = new RouteGraph();
foreach (string room in new[] { "Lobby", "Hall", "Lab", "Office", "Storage", "Remote" }) graph.AddVertex(room);
graph.SetEdge("Lobby", "Hall", 4);
graph.SetEdge("Hall", "Lab", 2);
graph.SetEdge("Hall", "Office", 3);
graph.SetEdge("Office", "Storage", 0);
string[] labels = graph.Vertices().OrderBy(r => r, StringComparer.Ordinal).ToArray();
int?[,] matrix = ToMatrix(graph, labels);
for (int i = 0; i < labels.Length; i++)
    for (int j = 0; j < labels.Length; j++)
    {
        bool exists = graph.TryGetWeight(labels[i], labels[j], out int weight);
        if (matrix[i, j].HasValue != exists || (exists && matrix[i, j] != weight))
            throw new Exception("Matrix mismatch.");
        if (matrix[i, j] != matrix[j, i]) throw new Exception("Symmetry mismatch.");
    }
int degreeSum = graph.Vertices().Sum(v => graph.Degree(v));
if (degreeSum != 2 * graph.EdgeCount || !graph.IsValid()) throw new Exception("Invariant mismatch.");
Console.WriteLine("All-pair presence, weights, symmetry, isolates, and degree sum passed.");

int?[,] ToMatrix(RouteGraph graph, string[] labels)
{
    var index = labels.Select((label, i) => (label, i)).ToDictionary(x => x.label, x => x.i);
    var matrix = new int?[labels.Length, labels.Length];
    foreach (string vertex in labels)
        foreach (var edge in graph.Neighbors(vertex))
            matrix[index[vertex], index[edge.Key]] = edge.Value;
    return matrix;
}

public class RouteGraph
{
    private readonly Dictionary<string, Dictionary<string, int>> adjacency
        = new(StringComparer.Ordinal);
    public int VertexCount => adjacency.Count;
    public int EdgeCount { get; private set; }

    public bool AddVertex(string vertex)
        => adjacency.TryAdd(vertex, new Dictionary<string, int>(StringComparer.Ordinal));

    public bool SetEdge(string a, string b, int minutes)
    {
        if (a == b) throw new ArgumentException("Self-loops are not allowed.");
        if (minutes < 0) throw new ArgumentOutOfRangeException(nameof(minutes));
        RequireVertex(a);
        RequireVertex(b);
        bool added = !adjacency[a].ContainsKey(b);
        adjacency[a][b] = minutes;
        adjacency[b][a] = minutes;
        if (added) EdgeCount++;
        return added;
    }

    public bool TryGetWeight(string a, string b, out int minutes)
    {
        minutes = default;
        return adjacency.TryGetValue(a, out var neighbors)
            && neighbors.TryGetValue(b, out minutes);
    }

    public int Degree(string vertex)
    {
        RequireVertex(vertex);
        return adjacency[vertex].Count;
    }

    public KeyValuePair<string, int>[] Neighbors(string vertex)
    {
        RequireVertex(vertex);
        return adjacency[vertex].ToArray();
    }

    public string[] Vertices() => adjacency.Keys.ToArray();

    public bool RemoveEdge(string a, string b)
    {
        if (!adjacency.TryGetValue(a, out var neighbors) || !neighbors.Remove(b)) return false;
        adjacency[b].Remove(a);
        EdgeCount--;
        return true;
    }

    public bool RemoveVertex(string vertex)
    {
        if (!adjacency.TryGetValue(vertex, out var neighbors)) return false;
        foreach (string neighbor in neighbors.Keys) adjacency[neighbor].Remove(vertex);
        EdgeCount -= neighbors.Count;
        adjacency.Remove(vertex);
        return true;
    }

    public bool IsValid()
    {
        long degreeSum = 0;
        foreach (var (vertex, neighbors) in adjacency)
        {
            degreeSum += neighbors.Count;
            foreach (var (neighbor, minutes) in neighbors)
            {
                if (vertex == neighbor || minutes < 0
                    || !adjacency.TryGetValue(neighbor, out var reverse)
                    || !reverse.TryGetValue(vertex, out int reverseMinutes)
                    || reverseMinutes != minutes) return false;
            }
        }
        return degreeSum == 2L * EdgeCount;
    }

    private void RequireVertex(string vertex)
    {
        if (!adjacency.ContainsKey(vertex)) throw new KeyNotFoundException($"Unknown vertex: {vertex}");
    }
}

All-pair presence, weights, symmetry, isolates, and degree sum passed.


## Project Connection

```{index} graph domain model
```

Identify a relationship in your course project. State its vertex and edge meanings, direction, weight units, loop/duplicate policy, and two operations that determine your representation choice.

## Submit

```{index} lab submission
```

Use the assignment submit panel when it is available in the published book.